# 07 — Behavioral v2: LOAO zero-day re-evaluation (Option A, Phase 3 continued)

LOAO (Leave-One-Attack-Out) re-evaluation of the `_behavioral_v2` models from
`06_behavioral_v2_retrain.ipynb`, on the **properly pipeline-integrated** population -- i.e. the
Phase 1 (`validate_behavioral_join.py`) row-hash-joined, dtype-normalized, BENIGN-unmatched-
dropped-and-documented data, reconstituted whole by concatenating
`cicids2017_behavioral_{train,val,test}.parquet` back together (2,026,398 rows: 1,600,657 BENIGN
+ all 425,741 attack rows, unchanged from before the join). This is **not** the same population
`05_loao_behavioral.ipynb` used (that notebook read `cicids2017_behavioral_labeled.parquet`
directly -- the standalone prototype, pre-Phase-1-join, full unreduced BENIGN, no row-hash-join
validation at all).

**Question this notebook answers**: `06_behavioral_v2_retrain.ipynb` found the Phase-1-reduced
BENIGN population (23.57% smaller) cost the Autoencoder some in-dataset F1 relative to the
standalone prototype (0.889 vs. 0.914). Does that same reduced population also cost it the
zero-day generalization gain the prototype found for PortScan/DDoS in `05_loao_behavioral.ipynb`
-- or does that gain hold up even on the smaller, properly-validated population?

**Per-fold checkpointing from the start** (not added after the fact this time -- the same design
that saved `05_loao_behavioral.ipynb`'s run from a mid-loop machine shutdown): each model's result
per category is written immediately to `reports/loao_behavioral_v2_checkpoints/{model}_{category}.json`;
a category with both checkpoints already present is skipped and reused on any re-run.

**Conventions reused directly from `05_loao_behavioral.ipynb`** (same fold-building logic, same
per-fold imputation/scaling, same pinned hyperparameters -- not re-tuned per fold):
- `build_loao_split`: train = all BENIGN except a held-out 20% + every attack category except the
  holdout one; zero-day test = that held-out BENIGN slice + only the holdout category's rows.
- Isolation Forest: `contamination=0.20` pinned (the value `06_behavioral_v2_retrain.ipynb`
  confirmed optimal), not re-swept per fold.
- Autoencoder: same architecture as `06_behavioral_v2_retrain.ipynb`; threshold = 99th percentile
  of *that fold's own* held-out BENIGN reconstruction error (never the zero-day test set), pinned
  from `06_behavioral_v2_retrain.ipynb`'s own confirmed-best percentile, not re-swept per fold.
- Fresh `MinMaxScaler` fit on each fold's full training data (all classes), same as
  `05_loao_behavioral.ipynb`.

`orig_row_index` (Phase 1's Fusion-Layer join key) is excluded from model input here -- not
needed for LOAO, which doesn't involve fusion against Random Forest.

## 1. Setup

In [1]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import json
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

import torch
import torch.nn as nn

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

print("torch version:", torch.__version__, " | cuda available:", torch.cuda.is_available())

torch version: 2.12.1+cu130  | cuda available: True


In [2]:
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

BEHAVIORAL_PROCESSED_DIR = ROOT / "data" / "processed" / "behavioral"
REPORTS_DIR = ROOT / "reports"

print("ROOT:", ROOT)

ROOT: E:\thesis


In [3]:
RANDOM_STATE = 42

IF_CONTAMINATION = 0.20
IF_N_ESTIMATORS = 100
AE_CHOSEN_PERCENTILE = 99

AE_HIDDEN_DIMS = [64, 32]
AE_BOTTLENECK_DIM = 16
AE_ACTIVATION = "relu"
AE_LEARNING_RATE = 1e-3
AE_EPOCHS = 30
AE_BATCH_SIZE = 8192
AE_WEIGHT_DECAY = 0.0
AE_EARLY_STOPPING_PATIENCE = 5
BENIGN_INTERNAL_VAL_FRAC = 0.10

LOAO_BENIGN_HOLDOUT_SIZE = 0.2

AE_CONFIG = {
    "hidden_dims": AE_HIDDEN_DIMS,
    "bottleneck_dim": AE_BOTTLENECK_DIM,
    "activation": AE_ACTIVATION,
    "learning_rate": AE_LEARNING_RATE,
    "epochs": AE_EPOCHS,
    "batch_size": AE_BATCH_SIZE,
    "weight_decay": AE_WEIGHT_DECAY,
    "early_stopping_patience": AE_EARLY_STOPPING_PATIENCE,
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

DEVICE: cuda


## 2. Reconstitute the full v2 population

Concatenates `cicids2017_behavioral_{train,val,test}.parquet` (Phase 1 output, before Phase 2's
scale/SMOTE) back together -- reconstructs exactly the full row-hash-joined, BENIGN-unmatched-
dropped population, since Phase 1 processed each split independently with the same join key/policy.

In [4]:
full_df = pd.concat([
    pd.read_parquet(BEHAVIORAL_PROCESSED_DIR / "cicids2017_behavioral_train.parquet"),
    pd.read_parquet(BEHAVIORAL_PROCESSED_DIR / "cicids2017_behavioral_val.parquet"),
    pd.read_parquet(BEHAVIORAL_PROCESSED_DIR / "cicids2017_behavioral_test.parquet"),
], ignore_index=True)

print("full_df shape:", full_df.shape)
full_df["Label"].value_counts()

full_df shape: (2026398, 92)


Label
BENIGN                        1600657
DoS Hulk                       172846
DDoS                           128014
PortScan                        90694
DoS GoldenEye                   10286
FTP-Patator                      5931
DoS slowloris                    5385
DoS Slowhttptest                 5228
SSH-Patator                      3219
Bot                              1948
Web Attack - Brute Force         1470
Web Attack - XSS                  652
Infiltration                       36
Web Attack - Sql Injection         21
Heartbleed                         11
Name: count, dtype: Int64

## 3. Feature selection + sentinel imputation helper

Same as `06_behavioral_v2_retrain.ipynb`'s `NON_FEATURE_COLS`, minus the CTI/MITRE columns (not
present in Phase 1's output -- those are Phase 2's join, not needed for LOAO evaluation) plus
`orig_row_index` excluded (Fusion-Layer join key, not a feature, not needed here).

In [5]:
NON_FEATURE_COLS = ["Label", "orig_row_index"]

def select_feature_columns(df, non_feature_cols=NON_FEATURE_COLS):
    return [
        c for c in df.columns
        if c not in non_feature_cols and pd.api.types.is_numeric_dtype(df[c])
    ]

SENTINEL_COLS = ["Init_Win_bytes_forward", "Init_Win_bytes_backward"]

def add_missing_flags_and_impute_fold(train_df_fold, test_df_fold, sentinel_cols=SENTINEL_COLS):
    train_df_fold = train_df_fold.copy()
    test_df_fold = test_df_fold.copy()
    for c in sentinel_cols:
        flag_col = f"{c}_missing"
        train_df_fold[flag_col] = train_df_fold[c].isna().astype(int)
        test_df_fold[flag_col] = test_df_fold[c].isna().astype(int)

        median = train_df_fold[c].median(skipna=True)
        train_df_fold[c] = train_df_fold[c].fillna(median)
        test_df_fold[c] = test_df_fold[c].fillna(median)
    return train_df_fold, test_df_fold

_preview_cols = select_feature_columns(full_df)
print(f"Candidate feature count (pre missing-flag): {len(_preview_cols)} (expected 90)")
assert len(_preview_cols) == 90

Candidate feature count (pre missing-flag): 90 (expected 90)


## 4. `build_loao_split` (reimplemented, same logic as `05_loao_behavioral.ipynb`)

In [6]:
def build_loao_split(df, holdout_category, benign_holdout_size=LOAO_BENIGN_HOLDOUT_SIZE, random_state=RANDOM_STATE):
    benign_mask = df["Label"] == "BENIGN"
    holdout_mask = df["Label"] == holdout_category

    other_attack_df = df.loc[~benign_mask & ~holdout_mask]
    holdout_df = df.loc[holdout_mask]

    benign_train_df, benign_zeroday_df = train_test_split(
        df.loc[benign_mask], test_size=benign_holdout_size, random_state=random_state
    )

    train_df_fold = pd.concat([benign_train_df, other_attack_df], ignore_index=True)
    zeroday_test_df_fold = pd.concat([benign_zeroday_df, holdout_df], ignore_index=True)
    return train_df_fold, zeroday_test_df_fold

## 5. Autoencoder architecture (identical to `06_behavioral_v2_retrain.ipynb`)

In [7]:
class Autoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dims, bottleneck_dim, activation="relu"):
        super().__init__()
        act_cls = {"relu": nn.ReLU, "tanh": nn.Tanh, "leaky_relu": nn.LeakyReLU}[activation]

        encoder_dims = [input_dim] + list(hidden_dims) + [bottleneck_dim]
        decoder_dims = [bottleneck_dim] + list(hidden_dims)[::-1] + [input_dim]

        encoder_layers = []
        for i in range(len(encoder_dims) - 1):
            encoder_layers.append(nn.Linear(encoder_dims[i], encoder_dims[i + 1]))
            if i < len(encoder_dims) - 2:
                encoder_layers.append(act_cls())
        self.encoder = nn.Sequential(*encoder_layers)

        decoder_layers = []
        for i in range(len(decoder_dims) - 1):
            decoder_layers.append(nn.Linear(decoder_dims[i], decoder_dims[i + 1]))
            if i < len(decoder_dims) - 2:
                decoder_layers.append(act_cls())
        self.decoder = nn.Sequential(*decoder_layers)

    def forward(self, x):
        return self.decoder(self.encoder(x))


def compute_reconstruction_error(model, X, device):
    model.eval()
    with torch.no_grad():
        X_tensor = torch.tensor(X, dtype=torch.float32).to(device)
        reconstructed = model(X_tensor)
        errors = torch.mean((reconstructed - X_tensor) ** 2, dim=1).cpu().numpy()
    return errors


def train_autoencoder(X_fit, X_internal_val, input_dim, config, device, verbose=False):
    model = Autoencoder(input_dim, config["hidden_dims"], config["bottleneck_dim"], config["activation"]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"], weight_decay=config["weight_decay"])
    criterion = nn.MSELoss()

    fit_tensor = torch.tensor(X_fit, dtype=torch.float32).to(device)
    internal_val_tensor = torch.tensor(X_internal_val, dtype=torch.float32).to(device)
    n_fit = fit_tensor.shape[0]
    batch_size = config["batch_size"]

    history = {"train_loss": [], "val_loss": []}
    best_val_loss = float("inf")
    best_state = None
    patience_counter = 0

    for epoch in range(config["epochs"]):
        model.train()
        perm = torch.randperm(n_fit, device=device)
        batch_losses = []
        for start in range(0, n_fit, batch_size):
            batch = fit_tensor[perm[start:start + batch_size]]
            optimizer.zero_grad()
            reconstructed = model(batch)
            loss = criterion(reconstructed, batch)
            loss.backward()
            optimizer.step()
            batch_losses.append(loss.item())
        train_loss = float(np.mean(batch_losses))

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(internal_val_tensor), internal_val_tensor).item()

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        if verbose:
            print(f"  Epoch {epoch + 1:3d}/{config['epochs']}  train_loss={train_loss:.6f}  val_loss={val_loss:.6f}")

        if val_loss < best_val_loss - 1e-6:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= config["early_stopping_patience"]:
                if verbose:
                    print(f"  Early stopping at epoch {epoch + 1} (best val_loss={best_val_loss:.6f})")
                break

    model.load_state_dict(best_state)
    return model, history

## 6. LOAO loop -- per-fold checkpointing from the start

Same 10 categories as `05_loao_behavioral.ipynb`. Each model's result for a category is written
immediately to `reports/loao_behavioral_v2_checkpoints/{model}_{category_slug}.json` after that
fold finishes. A category already fully checkpointed (both IF and AE files present) is skipped
and its cached result reused on any re-run.

In [8]:
CATEGORIES = [
    "DoS Hulk", "PortScan", "DDoS", "DoS GoldenEye", "FTP-Patator", "SSH-Patator",
    "DoS slowloris", "DoS Slowhttptest", "Bot", "Web Attack - Brute Force",
]

CHECKPOINT_DIR = REPORTS_DIR / "loao_behavioral_v2_checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

def slugify_category(cat):
    return re.sub(r"[\s\-]+", "_", cat.strip().lower())

def checkpoint_path(model_name, cat):
    return CHECKPOINT_DIR / f"{model_name.lower()}_{slugify_category(cat)}.json"

def save_checkpoint(model_name, result):
    path = checkpoint_path(model_name, result["category"])
    tmp_path = path.with_suffix(".json.tmp")
    with open(tmp_path, "w") as f:
        json.dump(result, f, indent=2)
    tmp_path.replace(path)

def load_checkpoint(model_name, cat):
    path = checkpoint_path(model_name, cat)
    if path.exists():
        with open(path) as f:
            return json.load(f)
    return None

_loop_start = time.perf_counter()

for cat in CATEGORIES:
    if_cached = load_checkpoint("IsolationForest", cat)
    ae_cached = load_checkpoint("Autoencoder", cat)
    if if_cached is not None and ae_cached is not None:
        print(f"{cat:30s}  SKIPPED (resumed from checkpoint): IF recall={if_cached['recall']:.4f} fpr={if_cached['fpr']:.4f}  "
              f"|  AE recall={ae_cached['recall']:.4f} fpr={ae_cached['fpr']:.4f}")
        continue

    _fold_start = time.perf_counter()
    train_df_fold, test_df_fold = build_loao_split(full_df, cat)

    train_df_fold, test_df_fold = add_missing_flags_and_impute_fold(train_df_fold, test_df_fold)
    fold_feature_cols = select_feature_columns(train_df_fold)
    assert fold_feature_cols == select_feature_columns(test_df_fold)
    assert len(fold_feature_cols) == 92

    missing_flag_cols = [f"{c}_missing" for c in SENTINEL_COLS]
    scale_cols = [c for c in fold_feature_cols if c not in missing_flag_cols]

    fold_scaler = MinMaxScaler()
    fold_scaler.fit(train_df_fold[scale_cols])
    train_df_fold = train_df_fold.copy()
    test_df_fold = test_df_fold.copy()
    train_df_fold[scale_cols] = fold_scaler.transform(train_df_fold[scale_cols])
    test_df_fold[scale_cols] = fold_scaler.transform(test_df_fold[scale_cols])

    X_train_fold = train_df_fold[fold_feature_cols].to_numpy(dtype=np.float64)
    X_test_fold = test_df_fold[fold_feature_cols].to_numpy(dtype=np.float64)
    y_true_fold = (test_df_fold["Label"] != "BENIGN").astype(int).to_numpy()

    if_model_fold = IsolationForest(
        n_estimators=IF_N_ESTIMATORS, contamination=IF_CONTAMINATION, random_state=RANDOM_STATE, n_jobs=-1,
    )
    if_model_fold.fit(X_train_fold)
    if_pred_fold = (if_model_fold.predict(X_test_fold) == -1).astype(int)

    if_tp = int(((if_pred_fold == 1) & (y_true_fold == 1)).sum())
    if_fn = int(((if_pred_fold == 0) & (y_true_fold == 1)).sum())
    if_fp = int(((if_pred_fold == 1) & (y_true_fold == 0)).sum())
    if_tn = int(((if_pred_fold == 0) & (y_true_fold == 0)).sum())
    if_recall = if_tp / (if_tp + if_fn) if (if_tp + if_fn) > 0 else float("nan")
    if_fpr = if_fp / (if_fp + if_tn) if (if_fp + if_tn) > 0 else float("nan")

    if_result = {
        "category": cat, "recall": if_recall, "fpr": if_fpr,
        "n_test_rows": len(test_df_fold), "n_attack_rows": int(y_true_fold.sum()),
        "n_benign_rows": int((y_true_fold == 0).sum()),
    }
    save_checkpoint("IsolationForest", if_result)

    fold_benign_train_df = train_df_fold[train_df_fold["Label"] == "BENIGN"]
    fold_benign_fit_df, fold_benign_internal_val_df = train_test_split(
        fold_benign_train_df, test_size=BENIGN_INTERNAL_VAL_FRAC, random_state=RANDOM_STATE
    )
    X_fold_fit = fold_benign_fit_df[fold_feature_cols].to_numpy(dtype=np.float32)
    X_fold_internal_val = fold_benign_internal_val_df[fold_feature_cols].to_numpy(dtype=np.float32)

    ae_model_fold, ae_history_fold = train_autoencoder(
        X_fold_fit, X_fold_internal_val, input_dim=len(fold_feature_cols), config=AE_CONFIG, device=DEVICE, verbose=False
    )

    X_test_fold_f32 = test_df_fold[fold_feature_cols].to_numpy(dtype=np.float32)
    fold_test_errors = compute_reconstruction_error(ae_model_fold, X_test_fold_f32, DEVICE)
    fold_internal_val_errors = compute_reconstruction_error(ae_model_fold, X_fold_internal_val, DEVICE)
    fold_threshold = float(np.percentile(fold_internal_val_errors, AE_CHOSEN_PERCENTILE))

    ae_pred_fold = (fold_test_errors > fold_threshold).astype(int)
    ae_tp = int(((ae_pred_fold == 1) & (y_true_fold == 1)).sum())
    ae_fn = int(((ae_pred_fold == 0) & (y_true_fold == 1)).sum())
    ae_fp = int(((ae_pred_fold == 1) & (y_true_fold == 0)).sum())
    ae_tn = int(((ae_pred_fold == 0) & (y_true_fold == 0)).sum())
    ae_recall = ae_tp / (ae_tp + ae_fn) if (ae_tp + ae_fn) > 0 else float("nan")
    ae_fpr = ae_fp / (ae_fp + ae_tn) if (ae_fp + ae_tn) > 0 else float("nan")

    ae_result = {
        "category": cat, "recall": ae_recall, "fpr": ae_fpr,
        "n_test_rows": len(test_df_fold), "n_attack_rows": int(y_true_fold.sum()),
        "n_benign_rows": int((y_true_fold == 0).sum()),
        "threshold": fold_threshold, "epochs_run": len(ae_history_fold["train_loss"]),
    }
    save_checkpoint("Autoencoder", ae_result)

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    fold_seconds = time.perf_counter() - _fold_start
    print(f"{cat:30s}  IF: recall={if_recall:.4f} fpr={if_fpr:.4f}  |  AE: recall={ae_recall:.4f} fpr={ae_fpr:.4f}  "
          f"(n_test={len(test_df_fold)}, {fold_seconds:.1f}s)")

loop_seconds = time.perf_counter() - _loop_start
print(f"\nFull LOAO loop ({len(CATEGORIES)} categories, 2 models) wall-clock time: {loop_seconds:.2f}s (device={DEVICE})")

loao_if_results = [load_checkpoint("IsolationForest", cat) for cat in CATEGORIES]
loao_ae_results = [load_checkpoint("Autoencoder", cat) for cat in CATEGORIES]
assert all(r is not None for r in loao_if_results), "Missing IsolationForest checkpoint(s) -- a fold failed silently"
assert all(r is not None for r in loao_ae_results), "Missing Autoencoder checkpoint(s) -- a fold failed silently"

loao_if_summary_df = pd.DataFrame(loao_if_results).sort_values("recall").reset_index(drop=True)
loao_ae_summary_df = pd.DataFrame(loao_ae_results).sort_values("recall").reset_index(drop=True)

DoS Hulk                        IF: recall=0.9779 fpr=0.1499  |  AE: recall=0.9488 fpr=0.0097  (n_test=492978, 23.7s)


PortScan                        IF: recall=0.1475 fpr=0.0860  |  AE: recall=0.7778 fpr=0.0095  (n_test=410826, 23.9s)


DDoS                            IF: recall=0.6385 fpr=0.1094  |  AE: recall=0.9706 fpr=0.0100  (n_test=448146, 22.8s)


DoS GoldenEye                   IF: recall=0.4206 fpr=0.0931  |  AE: recall=0.3213 fpr=0.0099  (n_test=330418, 22.5s)


FTP-Patator                     IF: recall=0.0005 fpr=0.0955  |  AE: recall=0.0005 fpr=0.0100  (n_test=326063, 23.1s)


SSH-Patator                     IF: recall=0.0000 fpr=0.0951  |  AE: recall=0.0009 fpr=0.0095  (n_test=323351, 21.9s)


DoS slowloris                   IF: recall=0.5822 fpr=0.0970  |  AE: recall=0.5848 fpr=0.0099  (n_test=325517, 22.6s)


DoS Slowhttptest                IF: recall=0.8374 fpr=0.0976  |  AE: recall=0.2592 fpr=0.0097  (n_test=325360, 22.2s)


Bot                             IF: recall=0.0226 fpr=0.0954  |  AE: recall=0.0205 fpr=0.0098  (n_test=322080, 22.3s)


Web Attack - Brute Force        IF: recall=0.0490 fpr=0.0963  |  AE: recall=0.0000 fpr=0.0097  (n_test=321602, 23.3s)

Full LOAO loop (10 categories, 2 models) wall-clock time: 228.21s (device=cuda)


In [9]:
print("=== Isolation Forest LOAO (behavioral_v2, 92 features, Phase 1+2 integrated) ===")
loao_if_summary_df

=== Isolation Forest LOAO (behavioral_v2, 92 features, Phase 1+2 integrated) ===


,category,recall,fpr,n_test_rows,n_attack_rows,n_benign_rows
0,SSH-Patator,0.000000,0.095130,323351,3219,320132
1,FTP-Patator,0.000506,0.095532,326063,5931,320132
2,Bot,0.022587,0.095448,322080,1948,320132
3,Web Attack - Brute Force,0.048980,0.096310,321602,1470,320132
4,PortScan,0.147452,0.085968,410826,90694,320132
5,DoS GoldenEye,0.420572,0.093133,330418,10286,320132
6,DoS slowloris,0.582173,0.097029,325517,5385,320132
7,DDoS,0.638547,0.109380,448146,128014,320132
8,DoS Slowhttptest,0.837414,0.097560,325360,5228,320132
9,DoS Hulk,0.977899,0.149929,492978,172846,320132


In [10]:
print("=== Autoencoder LOAO (behavioral_v2, 92 features, Phase 1+2 integrated) ===")
loao_ae_summary_df

=== Autoencoder LOAO (behavioral_v2, 92 features, Phase 1+2 integrated) ===


,category,recall,fpr,n_test_rows,n_attack_rows,n_benign_rows,threshold,epochs_run
0,Web Attack - Brute Force,0.000000,0.009705,321602,1470,320132,0.002009,30
1,FTP-Patator,0.000506,0.009965,326063,5931,320132,0.002506,30
2,SSH-Patator,0.000932,0.009521,323351,3219,320132,0.001559,30
3,Bot,0.020534,0.009752,322080,1948,320132,0.001414,30
4,DoS Slowhttptest,0.259181,0.009712,325360,5228,320132,0.001662,30
5,DoS GoldenEye,0.321311,0.009896,330418,10286,320132,0.001679,30
6,DoS slowloris,0.584773,0.009912,325517,5385,320132,0.002142,30
7,PortScan,0.777824,0.009487,410826,90694,320132,0.001349,30
8,DoS Hulk,0.948839,0.009674,492978,172846,320132,0.002191,30
9,DDoS,0.970574,0.010005,448146,128014,320132,0.001546,30


## 7. Three-way comparison: original 80-feature LOAO vs. standalone-prototype LOAO vs. v2-integrated LOAO

Focused on PortScan and DDoS -- the categories `05_loao_behavioral.ipynb`'s standalone prototype
found genuine zero-day gains for -- to answer the question this notebook exists to answer.

In [11]:
baseline_if_loao = pd.read_csv(REPORTS_DIR / "isolation_forest_loao_results.csv")
baseline_if_loao = baseline_if_loao[baseline_if_loao["status"] == "current_contamination_tuned"].set_index("category")
baseline_ae_loao = pd.read_csv(REPORTS_DIR / "autoencoder_loao_results.csv").set_index("category")

prototype_loao_df = pd.read_csv(REPORTS_DIR / "enhanced_ablation_loao_results.csv")
prototype_if_loao = prototype_loao_df[prototype_loao_df["model"] == "IsolationForest"].set_index("category")
prototype_ae_loao = prototype_loao_df[prototype_loao_df["model"] == "Autoencoder"].set_index("category")

v2_if_loao = loao_if_summary_df.set_index("category")
v2_ae_loao = loao_ae_summary_df.set_index("category")

FOCUS_CATEGORIES = ["PortScan", "DDoS"]

three_way_loao_rows = []
for model_name, baseline_idx, prototype_idx, v2_idx in [
    ("IsolationForest", baseline_if_loao, prototype_if_loao, v2_if_loao),
    ("Autoencoder", baseline_ae_loao, prototype_ae_loao, v2_ae_loao),
]:
    for cat in CATEGORIES:
        three_way_loao_rows.append({
            "model": model_name,
            "category": cat,
            "baseline_80f_recall": float(baseline_idx.loc[cat, "recall"]),
            "baseline_80f_fpr": float(baseline_idx.loc[cat, "fpr"]),
            "prototype_92f_recall": float(prototype_idx.loc[cat, "recall"]),
            "prototype_92f_fpr": float(prototype_idx.loc[cat, "fpr"]),
            "v2_integrated_92f_recall": float(v2_idx.loc[cat, "recall"]),
            "v2_integrated_92f_fpr": float(v2_idx.loc[cat, "fpr"]),
            "v2_vs_prototype_recall_delta": float(v2_idx.loc[cat, "recall"]) - float(prototype_idx.loc[cat, "recall"]),
            "v2_vs_baseline_recall_delta": float(v2_idx.loc[cat, "recall"]) - float(baseline_idx.loc[cat, "recall"]),
        })

three_way_loao_df = pd.DataFrame(three_way_loao_rows)

print("=== Focus categories: PortScan / DDoS ===")
focus_loao_df = three_way_loao_df[three_way_loao_df["category"].isin(FOCUS_CATEGORIES)]
print(focus_loao_df.to_string(index=False))

print("\n=== Full 10-category table ===")
print(three_way_loao_df.to_string(index=False))

=== Focus categories: PortScan / DDoS ===
          model category  baseline_80f_recall  baseline_80f_fpr  prototype_92f_recall  prototype_92f_fpr  v2_integrated_92f_recall  v2_integrated_92f_fpr  v2_vs_prototype_recall_delta  v2_vs_baseline_recall_delta
IsolationForest PortScan             0.002018          0.109562              0.462721           0.095021                  0.147452               0.085968                     -0.315269                     0.145434
IsolationForest     DDoS             0.634806          0.132316              0.636266           0.120823                  0.638547               0.109380                      0.002281                     0.003742
    Autoencoder PortScan             0.010960          0.009284              0.785510           0.009462                  0.777824               0.009487                     -0.007685                     0.766864
    Autoencoder     DDoS             0.555221          0.009241              0.969331           0.009653  

## 8. Save results

In [12]:
loao_if_summary_df["model"] = "IsolationForest"
loao_ae_summary_df["model"] = "Autoencoder"
combined_v2_loao_results_df = pd.concat([loao_if_summary_df, loao_ae_summary_df], ignore_index=True)

v2_loao_results_path = REPORTS_DIR / "behavioral_v2_loao_results.csv"
combined_v2_loao_results_df.to_csv(v2_loao_results_path, index=False)
print("Saved:", v2_loao_results_path)

v2_loao_comparison_path = REPORTS_DIR / "behavioral_v2_loao_three_way_comparison.csv"
three_way_loao_df.to_csv(v2_loao_comparison_path, index=False)
print("Saved:", v2_loao_comparison_path)

Saved: E:\thesis\reports\behavioral_v2_loao_results.csv
Saved: E:\thesis\reports\behavioral_v2_loao_three_way_comparison.csv


## 9. Conclusion

(Synthesized from the actual run results above.)

In [13]:
print("=== PortScan / DDoS: does the zero-day gain hold up on the smaller, join-validated population? ===\n")
for _, row in focus_loao_df.iterrows():
    print(f"{row['model']:16s} {row['category']:10s}  baseline_80f={row['baseline_80f_recall']:.4f}  "
          f"prototype_92f={row['prototype_92f_recall']:.4f}  v2_integrated_92f={row['v2_integrated_92f_recall']:.4f}  "
          f"(v2_vs_prototype={row['v2_vs_prototype_recall_delta']:+.4f}, v2_vs_baseline={row['v2_vs_baseline_recall_delta']:+.4f})")

print()
holds_up = (focus_loao_df["v2_vs_baseline_recall_delta"] > 0.05).sum()
n_focus_pairs = len(focus_loao_df)
print(f"PortScan/DDoS x {{IF, AE}} pairs still improved by >5% absolute zero-day recall vs. the 80-feature baseline: {holds_up} / {n_focus_pairs}")
print("\nSee reports/behavioral_v2_loao_three_way_comparison.csv for the full per-category table.")

=== PortScan / DDoS: does the zero-day gain hold up on the smaller, join-validated population? ===

IsolationForest  PortScan    baseline_80f=0.0020  prototype_92f=0.4627  v2_integrated_92f=0.1475  (v2_vs_prototype=-0.3153, v2_vs_baseline=+0.1454)
IsolationForest  DDoS        baseline_80f=0.6348  prototype_92f=0.6363  v2_integrated_92f=0.6385  (v2_vs_prototype=+0.0023, v2_vs_baseline=+0.0037)
Autoencoder      PortScan    baseline_80f=0.0110  prototype_92f=0.7855  v2_integrated_92f=0.7778  (v2_vs_prototype=-0.0077, v2_vs_baseline=+0.7669)
Autoencoder      DDoS        baseline_80f=0.5552  prototype_92f=0.9693  v2_integrated_92f=0.9706  (v2_vs_prototype=+0.0012, v2_vs_baseline=+0.4154)

PortScan/DDoS x {IF, AE} pairs still improved by >5% absolute zero-day recall vs. the 80-feature baseline: 3 / 4

See reports/behavioral_v2_loao_three_way_comparison.csv for the full per-category table.
